# ComfyUI AI Influencer on Kaggle

This notebook sets up ComfyUI for AI influencer generation on Kaggle with T4 x2 GPUs.

### Instructions:
1. Enable T4 x2 GPU.
2. Turn on Internet Access.
3. Run cells sequentially.

In [ ]:
!git clone https://github.com/Dhruwkd/comfyui-kaggle-influencer.git
%cd comfyui-kaggle-influencer
!chmod +x scripts/*.sh
!./scripts/install_nodes.sh
!./scripts/download_models.sh

In [ ]:
# Install cloudflared binary to a local path (avoid "Text file busy" on /usr/local/bin)
!mkdir -p /kaggle/working/bin
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/bin/cloudflared
!chmod +x /kaggle/working/bin/cloudflared

import subprocess
import threading
import time
import os
import sys
import re
import urllib.request
import urllib.error

# Verify ComfyUI exists
comfyui_path = "/kaggle/working/ComfyUI"
if not os.path.exists(comfyui_path):
    raise RuntimeError(f"ComfyUI not found at {comfyui_path}. Did cell 1 complete successfully?")

tunnel_url = {"value": None}
tunnel_ready = threading.Event()
tunnel_verified = threading.Event()

def run_tunnel():
    proc = subprocess.Popen(
        ["/kaggle/working/bin/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        line = line.strip()
        print(f"[cloudflared] {line}", flush=True)
        if "trycloudflare.com" in line and not tunnel_url["value"]:
            match = re.search(r'(https://[a-zA-Z0-9\-]+\.trycloudflare\.com)', line)
            if match:
                tunnel_url["value"] = match.group(1)
                tunnel_ready.set()

tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
tunnel_thread.start()

# Wait for tunnel URL to appear
print("⏳ Waiting for cloudflared tunnel URL...", flush=True)
tunnel_ready.wait(timeout=30)

if not tunnel_url["value"]:
    print("⚠️ Tunnel URL not detected. Check [cloudflared] logs above.", flush=True)
else:
    url = tunnel_url['value']
    print(f"\n🔗 Tunnel URL detected: {url}", flush=True)
    print("⏳ Verifying tunnel is accessible (this may take 10-20s)...", flush=True)
    
    # Verify tunnel actually works by hitting it
    for attempt in range(15):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            response = urllib.request.urlopen(req, timeout=10)
            if response.status == 200:
                print(f"✅ Tunnel verified and accessible!", flush=True)
                tunnel_verified.set()
                break
            elif response.status == 403:
                print(f"  Attempt {attempt+1}/15: Got 403 (Cloudflare Access), waiting...", flush=True)
            else:
                print(f"  Attempt {attempt+1}/15: Status {response.status}, waiting...", flush=True)
        except urllib.error.HTTPError as e:
            if e.code == 403:
                print(f"  Attempt {attempt+1}/15: Cloudflare Access (403), waiting...", flush=True)
            else:
                print(f"  Attempt {attempt+1}/15: HTTP {e.code}, waiting...", flush=True)
        except Exception as e:
            print(f"  Attempt {attempt+1}/15: {type(e).__name__}, waiting...", flush=True)
        time.sleep(2)
    
    if not tunnel_verified.is_set():
        print("⚠️ Tunnel not fully accessible yet. You may need to wait a bit more.", flush=True)
        print("   Try opening the URL in a minute, or refresh the page.", flush=True)

# Display the URL prominently
if tunnel_url["value"]:
    url = tunnel_url['value']
    print(f"\n{'#'*70}", flush=True)
    print(f"#  🌐  COMFYUI TUNNEL URL — COPY THIS LINK:", flush=True)
    print(f"#", flush=True)
    print(f"#      {url}", flush=True)
    print(f"#", flush=True)
    if tunnel_verified.is_set():
        print(f"#  ✅  TUNNEL VERIFIED — SHOULD WORK NOW", flush=True)
    else:
        print(f"#  ⏳  TUNNEL MAY NEED MORE TIME — WAIT 30s THEN TRY", flush=True)
    print(f"#  👆  OPEN THIS IN YOUR BROWSER (or Claude's Browser pane)", flush=True)
    print(f"#  ❌  DO NOT USE: http://localhost:8188  OR  http://0.0.0.0:8188", flush=True)
    print(f"{'#'*70}\n", flush=True)

# Give ComfyUI a moment to be ready
time.sleep(3)

# Start ComfyUI
os.chdir(comfyui_path)
print(f"🚀 Starting ComfyUI on port 8188 (bound to 0.0.0.0 for tunnel)...", flush=True)
if tunnel_url["value"]:
    print(f"📱 YOUR COMFYUI ADDRESS: {tunnel_url['value']}", flush=True)
!python main.py --lowvram --port 8188 --listen 0.0.0.0